## 训练契约补充

训练 batch 必须先移动到 device，再依次执行清梯度、前向、loss、反向和更新。验证使用 `inference_mode()`，测试阶段重新加载最佳验证 checkpoint。


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from common.engine import train_one_epoch, evaluate
from common.checkpoint import save_checkpoint, load_checkpoint
inputs = torch.randn(12, 4)
targets = (inputs[:, 0] > 0).long()
train_loader = DataLoader(TensorDataset(inputs, targets), batch_size=4)
validation_loader = DataLoader(TensorDataset(inputs, targets), batch_size=4)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1)
result = train_one_epoch(model, train_loader, loss_fn, optimizer, torch.device('cpu'))
scheduler.step()  # 该 scheduler 按 epoch 调用
assert result.examples == len(train_loader.dataset)
with torch.inference_mode():
    validation = evaluate(model, validation_loader, loss_fn, torch.device('cpu'))
assert torch.isfinite(torch.tensor(validation.loss))
print('weighted train loss:', result.loss, 'validation:', validation)

# 完整训练、验证与恢复训练

## 学习目标

能够解释一个 epoch 的执行顺序、指标聚合、最佳检查点、早停和恢复训练。


## 概念模型与执行路径

训练循环负责梯度更新，验证循环只测量泛化。平均损失必须按样本数加权。模型选择依据验证集，最终测试必须重新加载最佳检查点。


### 实验 1


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2


In [ ]:
import tempfile
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from common.engine import train_one_epoch, evaluate
from common.runtime import seed_everything

seed_everything(42)
inputs = torch.randn(128, 4)
labels = (inputs[:, 0] + inputs[:, 1] > 0).long()
train_loader = DataLoader(TensorDataset(inputs[:96], labels[:96]), batch_size=16, shuffle=True)
validation_loader = DataLoader(TensorDataset(inputs[96:], labels[96:]), batch_size=16)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = nn.CrossEntropyLoss()


### 实验 3


In [ ]:
from common.training import EarlyStopping
stopping = EarlyStopping(patience=2)
for epoch in range(1, 6):
    train_result = train_one_epoch(model, train_loader, loss_fn, optimizer, torch.device("cpu"))
    validation_result = evaluate(model, validation_loader, loss_fn, torch.device("cpu"))
    improved, should_stop = stopping.update(validation_result.accuracy)
    print(epoch, train_result, validation_result, "improved=", improved)
    if should_stop:
        break


### 实验 4


In [ ]:
from common.checkpoint import save_checkpoint, load_checkpoint
with tempfile.TemporaryDirectory() as directory:
    path = save_checkpoint(Path(directory) / "resume.pt", model, optimizer, epoch=epoch,
                           metrics={"validation_accuracy": validation_result.accuracy})
    restored = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
    restored_optimizer = torch.optim.Adam(restored.parameters(), lr=0.03)
    metadata = load_checkpoint(path, restored, restored_optimizer)
    torch.testing.assert_close(model(inputs[:4]), restored(inputs[:4]))
    print("restored metadata:", metadata)


## 底层机制

`model.eval()` 切换 Dropout/BatchNorm 行为，但不会关闭梯度；`inference_mode()` 关闭 autograd 并减少额外开销。检查点恢复 optimizer 是为了保留动量和自适应统计量。


## 检查点

指出训练一个 batch 的正确顺序：清梯度、前向、损失、反向、更新。解释为什么测试应加载最佳验证检查点。


## 试一试

故意把验证改为 `model.train()`，加入 Dropout 后重复评估三次，观察结果波动。


## 常见错误与调试

验证时更新参数、按 batch 平均后再简单平均不等长 batch、保存最后模型而非最佳模型、恢复模型却丢弃 optimizer 状态。
